# Embedding Model Bakeoff — CTI Corpus

Interactive version of `scripts/embedding_bakeoff.py` with per-step inspection and visualisations.

**Candidates**
| Model | Notes |
|---|---|
| `BAAI/bge-m3` | Dense + sparse + multi-vector; MTEB leader; production default |
| `thenlper/gte-large` | 1024-dim; preserves fine-grained ATT&CK subtechnique distinctions |
| `nomic-ai/nomic-embed-text-v1.5` | Apache 2.0; 8192-token context; requires `trust_remote_code=True` |

See `docs/rag/EMBEDDING_DECISION.md` for the formal decision record.

## 0  Environment

In [ ]:
import json
import random
import sys
import time
from pathlib import Path

import numpy as np

# Make src/ importable from the notebook
REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

from rag_cti.config import get_settings
settings = get_settings()
print("Settings loaded.")
print(f"  Embedding model (config default): {settings.embedding_model}")
print(f"  Groq key present : {bool(settings.groq_api_key.get_secret_value())}")

## 1  Parameters

In [ ]:
SAMPLE_SIZE = 200      # total corpus chunks to embed
N_QUERIES   = 20       # gold queries to generate (must be <= SAMPLE_SIZE)
TOP_K       = 5        # Recall@K
SEED        = 42
DEVICE      = None     # None = auto; 'cpu', 'cuda', 'mps'
SOURCES     = ("mitre", "otx")
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
OUT_PATH    = REPO_ROOT / "data" / "eval" / "bakeoff_results.json"

CANDIDATE_MODELS = [
    "BAAI/bge-m3",
    "thenlper/gte-large",
    "nomic-ai/nomic-embed-text-v1.5",
]

print(f"Corpus sample : {SAMPLE_SIZE} chunks from {SOURCES}")
print(f"Gold queries  : {N_QUERIES}")
print(f"Recall@K      : {TOP_K}")
print(f"Models        : {CANDIDATE_MODELS}")

## 2  Load corpus

In [ ]:
def _load_corpus(processed_dir: Path, sources: tuple, n: int, seed: int) -> list[dict]:
    rng = random.Random(seed)
    pool: list[dict] = []
    for src in sources:
        path = processed_dir / f"{src}.jsonl"
        if not path.exists():
            print(f"  WARNING: {path} not found — skipping")
            continue
        with path.open(encoding="utf-8") as fh:
            lines = [l for l in fh if l.strip()]
        for line in lines:
            pool.append(json.loads(line))
        print(f"  Loaded {src}: {len(lines)} chunks")
    if not pool:
        raise RuntimeError("No chunks found. Run seed scripts first.")
    rng.shuffle(pool)
    return pool[:n]

corpus = _load_corpus(PROCESSED_DIR, SOURCES, SAMPLE_SIZE, SEED)
print(f"\nCorpus size   : {len(corpus)} chunks")
print("Sample chunk  :", corpus[0]["content"][:120], "...")

## 3  Generate gold queries

Queries are generated by an LLM using a *fuzzy analyst* prompt — the model describes the "vibe" of the threat rather than quoting exact identifiers.  
Groq Llama is used for query generation; set `GROQ_API_KEY` before running this section.

In [ ]:
_QUERY_SYSTEM_PROMPT = (
    "Role: You are a seasoned Cyber Threat Intelligence (CTI) analyst working under pressure."
    "Context: You are searching a massive database for a specific piece of intelligence you vaguely remember, "
    "but you cannot recall the exact technical terms or indicators."
    "Task: Given a document chunk, generate ONE realistic search query that is intentionally difficult to retrieve. "
    "Follow these guidelines: "
    "Conceptual Overlap, not Keyword Overlap: Describe the intent, impact, or behavior without using the unique "
    "technical IDs, malware names, or specific CVEs found in the chunk. "
    "Analyst 'Memory Fog': Write the query as if you only remember the 'vibe' of the threat. "
    "Natural & Imperfect: Use natural language that might be slightly imprecise. "
    "Partial Info: Assume you only know one side of the story. "
    "Output ONLY the query text. No quotes, no preamble, no explanations. 15-30 words."
)

def generate_queries_groq(chunks: list[dict], key: str, model: str) -> list[str]:
    from groq import Groq
    client = Groq(api_key=key)
    queries = []
    for i, chunk in enumerate(chunks):
        snippet = chunk["content"][:1200]
        try:
            resp = client.chat.completions.create(
                model=model,
                max_tokens=64,
                messages=[
                    {"role": "system", "content": _QUERY_SYSTEM_PROMPT},
                    {"role": "user", "content": snippet},
                ],
            )
            queries.append((resp.choices[0].message.content or "").strip())
        except Exception as exc:
            print(f"  WARN chunk {i}: {exc}")
            queries.append(chunk["content"][:80])
        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{len(chunks)} queries generated")
    return queries


gold_chunks  = corpus[:N_QUERIES]
gold_indices = list(range(N_QUERIES))

groq_key = settings.groq_api_key.get_secret_value()
if not groq_key:
    raise RuntimeError("Set GROQ_API_KEY before generating bakeoff queries.")

print(f"Using Groq ({settings.groq_query_model}) for query generation ...")
queries = generate_queries_groq(gold_chunks, groq_key, settings.groq_query_model)
query_model_label = f"groq/{settings.groq_query_model}"

print(f"\nGenerated {len(queries)} queries via {query_model_label}")

### 3a  Inspect generated queries

Scan these before trusting the metrics — overly easy or overly generic queries skew results.

In [ ]:
for i, (q, chunk) in enumerate(zip(queries, gold_chunks)):
    print(f"[{i+1:02d}] Query   : {q}")
    print(f"      Gold    : {chunk['content'][:100]} ...")
    print()

## 4  Evaluate models

In [ ]:
from rag_cti.embeddings.embedder import Embedder

corpus_texts = [c["content"] for c in corpus]

def evaluate(model_name: str, device=None) -> dict:
    print(f"\n--- {model_name} ---")
    embedder = Embedder(model_name, device=device)

    t0 = time.perf_counter()
    corpus_vecs = embedder.encode(corpus_texts)
    t_corpus = time.perf_counter() - t0
    print(f"  Corpus encoded in {t_corpus:.1f}s  shape={corpus_vecs.shape}")

    t0 = time.perf_counter()
    query_vecs = embedder.encode(queries)
    t_query = time.perf_counter() - t0
    print(f"  Queries encoded in {t_query:.2f}s")

    scores  = query_vecs @ corpus_vecs.T
    top_idx = np.argsort(-scores, axis=1)[:, :TOP_K]

    hits, rr = 0, []
    for q_i, gold in enumerate(gold_indices):
        ranked = top_idx[q_i]
        if gold in ranked:
            rank = int(np.where(ranked == gold)[0][0]) + 1
            rr.append(1.0 / rank)
            hits += 1
        else:
            rr.append(0.0)

    return {
        "name": model_name,
        "dim": int(embedder.dimension),
        "recall_at_k": hits / len(queries),
        "mrr": float(np.mean(rr)),
        "corpus_encode_seconds": round(t_corpus, 3),
        "query_encode_seconds": round(t_query, 3),
        "seconds_per_query": round(t_query / len(queries), 4),
    }


results: list[dict] = []
for model_name in CANDIDATE_MODELS:
    try:
        results.append(evaluate(model_name, device=DEVICE))
    except Exception as exc:
        print(f"  FAILED: {exc}")

results.sort(key=lambda r: (-r["recall_at_k"], -r["mrr"]))
print("\nEvaluation complete.")

## 5  Save raw results

In [ ]:
from datetime import datetime

query_log = [
    {
        "query": queries[i],
        "gold_chunk_id": gold_chunks[i].get("id", ""),
        "gold_chunk_preview": gold_chunks[i].get("content", "")[:120],
        "generated_by": query_model_label,
    }
    for i in range(len(queries))
]

payload = {
    "timestamp": datetime.utcnow().isoformat(),
    "corpus_size": len(corpus),
    "n_queries": len(queries),
    "top_k": TOP_K,
    "query_generation_model": query_model_label,
    "sources": list(SOURCES),
    "queries": query_log,
    "results": results,
}

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Results saved to {OUT_PATH}")

## 6  Quick results table

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "Model": r["name"],
        "Dim": r["dim"],
        f"R@{TOP_K}": round(r["recall_at_k"], 3),
        "MRR": round(r["mrr"], 3),
        "s/query": r["seconds_per_query"],
    }
    for r in results
])
df.index += 1
df

## 7  Latency chart

In [ ]:
import matplotlib.pyplot as plt

names   = [r["name"].split("/")[-1] for r in results]
latency = [r["seconds_per_query"] for r in results]

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(names, latency, color="seagreen")
ax.set_title("Inference latency (s/query)")
ax.set_ylabel("Seconds")
ax.tick_params(axis="x", rotation=15)
for bar, val in zip(bars, latency):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(val * 0.01, 0.0002),
            f"{val:.4f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 8  Automated Report

**Idempotent** — loads `data/eval/bakeoff_results.json` and (re)writes `docs/rag/EMBEDDING_DECISION.md`.  
Can be run independently of sections 0–7 after a bakeoff run.

**Decision rule:** MRR is the primary metric (top-heavy ranking reduces analyst cognitive load); Recall@5 is the tie-breaker.  
**Dependency:** `pip install tabulate` is required for `df.to_markdown()`.

In [ ]:
# ── Section 8: Automated Report ──────────────────────────────────────────────
# Idempotent: loads bakeoff_results.json; (re)writes EMBEDDING_DECISION.md.
# Can be run independently of sections 0-7.
# Requires: pip install tabulate  (for df.to_markdown)

import json
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

_REPO_ROOT    = Path().resolve().parent
_RESULTS_PATH = _REPO_ROOT / "data" / "eval" / "bakeoff_results.json"
_CHART_PATH   = _REPO_ROOT / "data" / "eval" / "bakeoff_chart.png"
_DECISION_DOC = _REPO_ROOT / "docs" / "rag" / "EMBEDDING_DECISION.md"

if not _RESULTS_PATH.exists():
    print(f"ERROR: {_RESULTS_PATH} not found.")
    print("Run sections 0-7 first, then re-run this cell.")
else:
    saved      = json.loads(_RESULTS_PATH.read_text(encoding="utf-8"))
    top_k      = saved["top_k"]
    recall_col = f"Recall@{top_k}"

    # ── 1. Build DataFrame ────────────────────────────────────────────────────
    rows = []
    for r in saved["results"]:
        rows.append({
            "Model":     r["name"].split("/")[-1],
            "Full Name": r["name"],
            "Dim":       r["dim"],
            recall_col:  round(r["recall_at_k"], 4),
            "MRR":       round(r["mrr"], 4),
            "s/query":   r["seconds_per_query"],
        })
    df = pd.DataFrame(rows)

    # ── 2. Seaborn grouped bar chart: Recall@K vs MRR side-by-side ───────────
    plot_df = df.melt(
        id_vars="Model",
        value_vars=[recall_col, "MRR"],
        var_name="Metric",
        value_name="Value",
    )

    sns.set_theme(style="whitegrid", font_scale=1.05)
    fig, ax = plt.subplots(figsize=(10, 5))
    palette = {recall_col: "#4C72B0", "MRR": "#DD8452"}

    sns.barplot(
        data=plot_df,
        x="Model",
        y="Value",
        hue="Metric",
        palette=palette,
        ax=ax,
        edgecolor="white",
        width=0.6,
    )

    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=3, fontsize=9)

    ax.set_ylim(0, 1.15)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.set_title(
        f"CTI Embedding Bakeoff — {recall_col} vs MRR  "
        f"({saved['n_queries']} queries, corpus={saved['corpus_size']})",
        fontsize=12,
        pad=12,
    )
    ax.set_xlabel("Model", fontsize=11)
    ax.set_ylabel("Score", fontsize=11)
    ax.legend(title="Metric", bbox_to_anchor=(1.01, 1), loc="upper left")
    sns.despine()
    plt.tight_layout()
    _CHART_PATH.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(_CHART_PATH, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Chart saved -> {_CHART_PATH}")

    # ── 3. Decision logic: primary = MRR, tie-break = Recall@K ───────────────
    ranked = df.sort_values(
        by=["MRR", recall_col], ascending=False
    ).reset_index(drop=True)

    winner = ranked.iloc[0]
    print(f"\nRanking  (primary: MRR desc, tie-break: {recall_col} desc)")
    for _, row in ranked.iterrows():
        tag = "  WINNER" if row["Full Name"] == winner["Full Name"] else ""
        print(
            f"  {row['Full Name']:<45}  "
            f"MRR={row['MRR']:.4f}  "
            f"{recall_col}={row[recall_col]:.4f}{tag}"
        )

    # ── 4. Markdown table via df.to_markdown() ────────────────────────────────
    table_df = ranked[["Model", "Dim", recall_col, "MRR", "s/query"]].copy()
    ranks = ["**Winner**"] + [f"#{i + 2}" for i in range(len(table_df) - 1)]
    table_df.insert(0, "Rank", ranks)
    md_table = table_df.to_markdown(index=False)

    # ── 5. Write EMBEDDING_DECISION.md ───────────────────────────────────────
    now_str   = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    w_name    = winner["Full Name"]
    w_dim     = int(winner["Dim"])
    w_recall  = winner[recall_col]
    w_mrr     = winner["MRR"]
    w_latency = winner["s/query"]

    doc_lines = [
        "# Embedding Model Decision",
        "",
        f"> Auto-generated {now_str} from `data/eval/bakeoff_results.json`.  ",
        "> Re-run notebook cell **8 — Automated Report** to refresh after a new bakeoff run.",
        "",
        "---",
        "",
        "## 1. Decision",
        "",
        "| Field | Value |",
        "|---|---|",
        f"| **Chosen model** | `{w_name}` |",
        f"| **HuggingFace ID** | `{w_name}` |",
        f"| **Vector dimension** | {w_dim} |",
        f"| **{recall_col} on CTI corpus** | {w_recall:.4f} |",
        f"| **MRR on CTI corpus** | {w_mrr:.4f} |",
        f"| **Seconds per query** | {w_latency:.4f} |",
        f"| **Decision date** | {now_str} |",
        "",
        "Set the chosen model in `.env`:",
        "",
        "```",
        f"EMBEDDING_MODEL={w_name}",
        "```",
        "",
        "Then rebuild the Qdrant index:",
        "",
        "```bash",
        "python scripts/ingest.py --sources mitre otx pdfs",
        "```",
        "",
        "---",
        "",
        "## 2. All Model Results",
        "",
        md_table,
        "",
        f"*Ranked by MRR descending; {recall_col} used as tie-breaker.*",
        "",
        "---",
        "",
        "## 3. Selection Logic",
        "",
        "**Primary metric: MRR (Mean Reciprocal Rank)**",
        "",
        "MRR was chosen as the primary selection criterion because it rewards models that place",
        "the most relevant CTI chunk **at the very top** of the ranked list.",
        "",
        "In analyst triage workflows, the first result carries disproportionate weight:",
        "",
        "- Analysts under time pressure rarely scroll past the first two results.",
        f"- A model with high {recall_col} but low MRR retrieves the right chunk *somewhere*",
        f"  in the top-{top_k}, but may bury it at rank 4 or 5.",
        "- A model with high MRR consistently surfaces the gold chunk at rank 1-2,",
        "  **reducing cognitive load and false-negative triage decisions**.",
        "",
        f"**Tie-breaker: {recall_col}**",
        "",
        f"When two models share the same MRR, {recall_col} is used as a tie-breaker:",
        "it captures whether the model surfaces the relevant chunk within a broader candidate set,",
        "which matters for downstream re-ranking stages.",
        "",
        "---",
        "",
        "## 4. Final Recommendation",
        "",
        f"> **Use `{w_name}`** as the production embedding model.",
        "",
        f"It achieved the highest MRR ({w_mrr:.4f}) on the CTI evaluation corpus",
        f"({saved['n_queries']} fuzzy analyst queries over {saved['corpus_size']} chunks),",
        "meaning it consistently ranks the most relevant threat intelligence chunk closest",
        "to the top of the result list — the property that most directly reduces analyst triage time.",
    ]

    _DECISION_DOC.parent.mkdir(parents=True, exist_ok=True)
    _DECISION_DOC.write_text("\n".join(doc_lines) + "\n", encoding="utf-8")
    print(f"\nDecision doc saved -> {_DECISION_DOC}")